# LIMPEZA E PADRONIZAÇÃO

Importação das bibliotecas e leituda dos dados

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

df_olympics_1896_2022 = pd.read_csv('raw/olympics_historico.csv')
df_olympics_2024 = pd.read_csv('raw/olympics_paris2024.csv')

C:\Users\cpc12\AppData\Local\Temp\ipykernel_9104\1543068148.py:5: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_olympics_1896_2022 = pd.read_csv('raw/olympics_historico.csv')


## Jogos Olímpicos de 1896 até 2022

Remoção dos Jogos Olímpicos Intercalados de 1906 (COI hoje não os reconhece)

In [3]:
df_olympics_1896_2022 = df_olympics_1896_2022[df_olympics_1896_2022['edition'] != 1906]

Desduplicamos medalhas de equipes e padronizamos os códigos dos países (NOC)

In [4]:
df_olympics_1896_2022 = df_olympics_1896_2022.drop_duplicates(subset=['edition', 'country_noc', 'sport', 'event', 'medal'])

# Padronizar códigos dos países (NOC)
mapeamento_noc = {
    'EUA': 'GER',  # Equipe Alemã Unida
    'URS': 'RUS',  # União Soviética para Rússia
}
df_olympics_1896_2022['country_noc'] = df_olympics_1896_2022['country_noc'].replace(mapeamento_noc)


Criando o Sexo do "olympic_historico.csv"

Se cria uma nova coluna chamada sexo, pegando tudo após a vírgula na coluna event e removendo espaços extras.
Assim, para "Skeleton, Men" o resultado será "Men", e para "4 × 100 metres Medley Relay, Men" também será "Men".

In [5]:
df_olympics_1896_2022['sexo'] = (
    df_olympics_1896_2022['event']
    .str.split(',')
    .str[-1]
    .str.strip()
)

Padronizando as colunas do "olympic_historico.csv" e eliminar as ocorrências que n ganharam medalhas

In [6]:
# Padronização do olympics_historico.csv para integração
df_olympics_1896_2022 = df_olympics_1896_2022.rename(columns={
    'edition': 'edition',
    'country_noc': 'country_noc',
    'sport': 'sport',
    'event': 'event',
    'medal': 'medal',
})

df_olympics_1896_2022 = df_olympics_1896_2022.dropna(subset=['medal'])

df_olympics_1896_2022 = df_olympics_1896_2022.drop(columns=['edition_id', 'result_id', 'position', 'is_team_sport'])

df_olympics_1896_2022

,edition,country_noc,sport,event,athlete,athlete_id,medal,sexo
272147,1928 Winter Olympics,USA,Skeleton,"Skeleton, Men",Jennison Heaton,86530,Gold,Men
272148,1948 Winter Olympics,ITA,Skeleton,"Skeleton, Men",Nino Bibbia,84107,Gold,Men
272149,2002 Winter Olympics,USA,Skeleton,"Skeleton, Men","Jim Shea, Jr.",101695,Gold,Men
272150,2002 Winter Olympics,USA,Skeleton,"Skeleton, Women",Tristan Gale,101682,Gold,Women
272151,2006 Winter Olympics,CAN,Skeleton,"Skeleton, Men",Duff Gibson,101704,Gold,Men
...,...,...,...,...,...,...,...,...
316828,2022 Winter Olympics,USA,Snowboarding,"Slopestyle, Women",Julia Marino,138364,Silver,Women
316829,2022 Winter Olympics,ESP,Snowboarding,"Halfpipe, Women",Queralt Castellet,109979,Silver,Women
316830,2022 Winter Olympics,FRA,Snowboarding,"Cross, Women",Chloé Trespeuch,127764,Silver,Women
316831,2022 Winter Olympics,NZL,Snowboarding,"Big Air, Women",Zoi Sadowski-Synnott,137770,Silver,Women


## Jogos Olímpicos de Paris 2024

Padronizar o "olympics_paris2024.csv"

In [7]:
# Renomear colunas para o padrão do histórico
df_olympics_2024 = df_olympics_2024.rename(columns={
    'country_code': 'country_noc',
    'medal_type': 'medal',
    'discipline': 'sport',
    'gender': 'sexo',
    'name': 'athlete',
    'code_athlete': 'athlete_id'
})

# Definir edição fixa
df_olympics_2024['edition'] = '2024 Summer Olympics'

df_olympics_2024 = df_olympics_2024.drop(columns=[
    'medal_date',
    'medal_code',
    'country',
    'country_long',
    'nationality_code',
    'nationality',
    'nationality_long',
    'team',
    'team_gender',
    'event_type',
    'url_event',
    'birth_date',
    'code_team',
    'is_medallist'
])

df_olympics_2024 = df_olympics_2024[['edition', 'country_noc', 'sport', 'event', 'athlete', 'athlete_id', 'medal', 'sexo']]

Eliminar o medal dentro da coluna "medal"

In [8]:
df_olympics_2024['medal'] = df_olympics_2024['medal'].str.replace(' Medal', '', regex=False)

Deixer a coluna "sexo" no padrão

In [9]:
df_olympics_2024['sexo'] = df_olympics_2024['sexo'].replace({
    'Male': 'Men',
    'Female': 'Women'
})

Inverter o formato "SOBRENOME Nome" para "Nome Sobrenome" na coluna athlete do DataFrame df_olympics_2024:

In [10]:
def inverter_nome(nome):
    partes = nome.split()
    if len(partes) > 1:
        return ' '.join(partes[1:]) + ' ' + partes[0].capitalize()
    return nome

df_olympics_2024['athlete'] = df_olympics_2024['athlete'].apply(inverter_nome)

In [11]:
df_olympics_2024

,edition,country_noc,sport,event,athlete,athlete_id,medal,sexo
0,2024 Summer Olympics,BEL,Cycling Road,Men's Individual Time Trial,Remco Evenepoel,1903136,Gold,Men
1,2024 Summer Olympics,ITA,Cycling Road,Men's Individual Time Trial,Filippo Ganna,1923520,Silver,Men
2,2024 Summer Olympics,BEL,Cycling Road,Men's Individual Time Trial,AERT Wout Van,1903147,Bronze,Men
3,2024 Summer Olympics,AUS,Cycling Road,Women's Individual Time Trial,Grace Brown,1940173,Gold,Women
4,2024 Summer Olympics,GBR,Cycling Road,Women's Individual Time Trial,Anna Henderson,1912525,Silver,Women
...,...,...,...,...,...,...,...,...
2310,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Martina Centofanti,1923700,Bronze,Women
2311,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Agnese Duranti,1923701,Bronze,Women
2312,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Alessia Maurelli,1923699,Bronze,Women
2313,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Daniela Mogurean,1923703,Bronze,Women


# INTEGRAÇÃO DOS DADOS

Comparar os tipos de dados das colunas dos dois DataFrames

In [12]:
pd.DataFrame({
    '1896_2022': df_olympics_1896_2022.dtypes,
    '2024': df_olympics_2024.dtypes
})

,1896_2022,2024
edition,object,object
country_noc,object,object
sport,object,object
event,object,object
athlete,object,object
athlete_id,int64,int64
medal,object,object
sexo,object,object


Padronizar cada DataFrame separadamente

In [13]:
df_olympics_1896_2022['country_noc'] = df_olympics_1896_2022['country_noc'].str.upper().str.strip()
df_olympics_1896_2022['medal'] = df_olympics_1896_2022['medal'].str.capitalize().str.strip()
df_olympics_1896_2022['sexo'] = df_olympics_1896_2022['sexo'].str.capitalize().str.strip()
df_olympics_1896_2022['sport'] = df_olympics_1896_2022['sport'].str.strip()

df_olympics_2024['country_noc'] = df_olympics_2024['country_noc'].str.upper().str.strip()
df_olympics_2024['medal'] = df_olympics_2024['medal'].str.capitalize().str.strip()
df_olympics_2024['sexo'] = df_olympics_2024['sexo'].str.capitalize().str.strip()
df_olympics_2024['sport'] = df_olympics_2024['sport'].str.strip()

Unir os dois Dataframes

In [14]:
df_olympics = pd.concat([df_olympics_1896_2022, df_olympics_2024], ignore_index=True)
df_olympics

,edition,country_noc,sport,event,athlete,athlete_id,medal,sexo
0,1928 Winter Olympics,USA,Skeleton,"Skeleton, Men",Jennison Heaton,86530,Gold,Men
1,1948 Winter Olympics,ITA,Skeleton,"Skeleton, Men",Nino Bibbia,84107,Gold,Men
2,2002 Winter Olympics,USA,Skeleton,"Skeleton, Men","Jim Shea, Jr.",101695,Gold,Men
3,2002 Winter Olympics,USA,Skeleton,"Skeleton, Women",Tristan Gale,101682,Gold,Women
4,2006 Winter Olympics,CAN,Skeleton,"Skeleton, Men",Duff Gibson,101704,Gold,Men
...,...,...,...,...,...,...,...,...
22694,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Martina Centofanti,1923700,Bronze,Women
22695,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Agnese Duranti,1923701,Bronze,Women
22696,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Alessia Maurelli,1923699,Bronze,Women
22697,2024 Summer Olympics,ITA,Rhythmic Gymnastics,Group All-Around,Daniela Mogurean,1923703,Bronze,Women


# CARGA

## Medalhas por país e edição "medalhas_1986_2024.csv".

In [59]:
# filtrar apenas linhas com medalha (segurança)
df_medalhas = df_olympics[df_olympics['medal'].notna()]

# criar tabela dinâmica
tabela = df_medalhas.pivot_table(
    index=['edition', 'country_noc'],
    columns='medal',
    aggfunc='size',
    fill_value=0
)

# garantir colunas
for col in ['Gold', 'Silver', 'Bronze']:
    if col not in tabela.columns:
        tabela[col] = 0

# renomear colunas
tabela = tabela.rename(columns={
    'Gold': 'gold',
    'Silver': 'silver',
    'Bronze': 'bronze'
})

# criar total
tabela['total'] = tabela[['gold', 'silver', 'bronze']].sum(axis=1)

# voltar índice para colunas
df_medal = tabela.reset_index()

# ordenar (opcional)
df_medal = df_medal.sort_values(by='total', ascending=False)

df_medal

medal,edition,country_noc,bronze,gold,silver,total
1878,2024 Summer Olympics,USA,95,134,101,330
35,1904 Summer Olympics,USA,80,79,85,244
752,1980 Summer Olympics,RUS,46,80,69,195
1821,2024 Summer Olympics,FRA,39,53,95,187
823,1984 Summer Olympics,USA,30,82,61,173
...,...,...,...,...,...,...
1256,2004 Summer Olympics,HKG,0,0,1,1
1222,2002 Winter Olympics,SLO,1,0,0,1
1238,2004 Summer Olympics,CMR,0,1,0,1
1246,2004 Summer Olympics,ERI,1,0,0,1


Salvar em CSV.

In [60]:
df_medal.to_csv('bronze/medalhas_1986_2024.csv', index=False)

## Modalidades esportivas por edição "modalidades_1986_2024.csv".

In [15]:
df_sport = df_olympics[['edition', 'sport', 'event','athlete_id']].drop_duplicates()

df_sport

,edition,sport,event,athlete_id
0,1928 Winter Olympics,Skeleton,"Skeleton, Men",86530
1,1948 Winter Olympics,Skeleton,"Skeleton, Men",84107
2,2002 Winter Olympics,Skeleton,"Skeleton, Men",101695
3,2002 Winter Olympics,Skeleton,"Skeleton, Women",101682
4,2006 Winter Olympics,Skeleton,"Skeleton, Men",101704
...,...,...,...,...
22694,2024 Summer Olympics,Rhythmic Gymnastics,Group All-Around,1923700
22695,2024 Summer Olympics,Rhythmic Gymnastics,Group All-Around,1923701
22696,2024 Summer Olympics,Rhythmic Gymnastics,Group All-Around,1923699
22697,2024 Summer Olympics,Rhythmic Gymnastics,Group All-Around,1923703


In [16]:
df_sport.to_csv('bronze/modalidades_1986_2024.csv', index=False)

## Distribuição de atletas por sexo, país e edição "atletas_por_sexo.csv".

In [77]:
df_athlete_sexo = df_olympics[['athlete', 'athlete_id', 'country_noc', 'sexo', 'sport', 'medal', 'edition']]

df_athlete_sexo

,athlete,athlete_id,country_noc,sexo,sport,medal,edition
0,Jennison Heaton,86530,USA,Men,Skeleton,Gold,1928 Winter Olympics
1,Nino Bibbia,84107,ITA,Men,Skeleton,Gold,1948 Winter Olympics
2,"Jim Shea, Jr.",101695,USA,Men,Skeleton,Gold,2002 Winter Olympics
3,Tristan Gale,101682,USA,Women,Skeleton,Gold,2002 Winter Olympics
4,Duff Gibson,101704,CAN,Men,Skeleton,Gold,2006 Winter Olympics
...,...,...,...,...,...,...,...
22694,Martina Centofanti,1923700,ITA,Women,Rhythmic Gymnastics,Bronze,2024 Summer Olympics
22695,Agnese Duranti,1923701,ITA,Women,Rhythmic Gymnastics,Bronze,2024 Summer Olympics
22696,Alessia Maurelli,1923699,ITA,Women,Rhythmic Gymnastics,Bronze,2024 Summer Olympics
22697,Daniela Mogurean,1923703,ITA,Women,Rhythmic Gymnastics,Bronze,2024 Summer Olympics


In [78]:
df_athlete_sexo.to_csv('bronze/atletas_por_sexo.csv', index=False)